# Charts for the Olist late-delivery project

SQL already did the thinking (joins, late definition, ranks). This notebook only pulls those query results into pandas and draws three charts.

I keep the password in `../.env` so it does not sit in this file.

In [ ]:
from pathlib import Path
import os

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from dotenv import load_dotenv
from sqlalchemy import create_engine

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

load_dotenv(ROOT / ".env")
engine = create_engine(os.environ["DATABASE_URL"])

sns.set_theme(style="whitegrid", font_scale=1.1)
plt.rcParams["figure.facecolor"] = "white"

def read_sql_file(name):
    # PostgreSQL ignores -- comments. I keep explanations in the .sql files.
    sql = (ROOT / "sql" / name).read_text(encoding="utf-8")
    return pd.read_sql(sql, engine)

print("connected")

## 1. Late rate by state

I expect the northeast to be worse than São Paulo, because most Olist sellers sit in the southeast. If SP is also the worst, my join is probably wrong.

In [ ]:
states = read_sql_file("02_late_by_state.sql")
display(states.head(10))

# worst 10, plus SP so I can compare the northeast to a "normal" state
top_states = pd.concat(
    [states.head(10), states[states["customer_state"] == "SP"]]
).drop_duplicates("customer_state")
fig, ax = plt.subplots(figsize=(9, 6))
sns.barplot(
    data=top_states,
    x="late_rate_pct",
    y="customer_state",
    color="#c44e52",
    ax=ax,
)
ax.set_xlabel("Late deliveries (%)")
ax.set_ylabel("Customer state")
ax.set_title("Northeast states wait longer than São Paulo")
fig.tight_layout()
fig.savefig(ROOT / "figures" / "late_rate_by_state.png", dpi=160)
plt.show()

## 2. Late rate by category

Smaller differences than states. I still plot the worst 12 so I can talk about them in the README.

In [ ]:
categories = read_sql_file("03_late_by_category.sql")
display(categories.head(10))

top_cats = categories.head(12)
fig, ax = plt.subplots(figsize=(9, 6))
sns.barplot(
    data=top_cats,
    x="late_rate_pct",
    y="category",
    color="#4c72b0",
    ax=ax,
)
ax.set_xlabel("Late deliveries (%)")
ax.set_ylabel("Product category")
ax.set_title("Audio and electronics are late more often than housewares")
fig.tight_layout()
fig.savefig(ROOT / "figures" / "late_rate_by_category.png", dpi=160)
plt.show()

## 3. Reviews: late vs on time

If late and on-time scores are almost equal, delay does not really hurt the rating. If they are far apart, it does.

In [ ]:
reviews = read_sql_file("04_late_vs_reviews.sql")
display(reviews)

plot_df = reviews.copy()
plot_df["delivery"] = plot_df["delivery"].replace({"late": "Late", "on_time": "On time"})

fig, ax = plt.subplots(figsize=(7, 5))
sns.barplot(
    data=plot_df,
    x="delivery",
    y="avg_review_score",
    hue="delivery",
    palette={"Late": "#c44e52", "On time": "#55a868"},
    legend=False,
    ax=ax,
)
ax.set_xlabel("")
ax.set_ylabel("Average review score (1–5)")
ax.set_ylim(0, 5)
ax.set_title("Late orders are rated almost 2 stars lower")
fig.tight_layout()
fig.savefig(ROOT / "figures" / "review_score_late_vs_ontime.png", dpi=160)
plt.show()

## What I take from the three charts

- Geography moves the late rate a lot (AL ~24% vs SP ~6%).
- Category moves it less (audio ~13% vs housewares ~6%).
- Reviews move a lot: 2.57 vs 4.29. That is the number I would put at the top of a short writeup.

Limitations I already wrote in the SQL files: delivered orders only, category is item-level, lateness and a bad product can happen together.